# Annotation Analysis — MEL-1 campaign

Analysis of the human annotations of the *Multimodal Entity Linking* dataset.
Annotation task: for each instance (a **mention** + an **image** + an **entity**),
the annotator answers `entity_is_illustrated` → **YES / NO**.

All loading, statistics, HTML display and figure code lives in the sibling
modules (`loading.py`, `stats.py`, `display.py`, `plots.py`); this notebook is
the narrative that calls them. It is **generic over N annotators**: drop a new
`<user>/user_state.json` folder under the annotation directory and rerun.

**Outline**
- §0 — Setup & data loading
- §1 — Master table
- §2 — General statistics
- §3 — Per-annotator statistics (bias, Cohen's κ, confusion)
- §4 — Per-category statistics (PERS / ORG / LOC)
- §5 — Category × annotator
- §6 — Extreme cases (disagreements, fragile, unanimous)
- §7 — Annotation time (behavioral logs)

## §0 — Setup & data loading

In [ ]:
import pandas as pd
import plots
import stats
from display import show_cards, show_table
from IPython.display import display
from loading import (
    DEFAULT_ANNOT_DIR,
    DEFAULT_DATA_DIR,
    build_master,
    load_annotations,
    load_decision_times,
    load_items,
    load_kb_maps,
)

from fig_gen.utils import apply_style  # on sys.path via the modules above

apply_style()
pd.set_option("display.max_colwidth", 200)

# Point these at another campaign to reanalyze it.
DATA_DIR = DEFAULT_DATA_DIR
ANNOT_DIR = DEFAULT_ANNOT_DIR

In [ ]:
kb = load_kb_maps(DATA_DIR / "kb.jsonl")
items = load_items(DATA_DIR, kb)
annotations = load_annotations(ANNOT_DIR)
ANNOTATORS = sorted(annotations)
print(f"{len(kb):,} KB entities | {len(items):,} items | "
      f"annotators: {ANNOTATORS}")

## §1 — Master table

One row per annotated instance: item fields, one label column per annotator,
and vote aggregates (`n_yes`/`n_no`, `majority`, `disagree`, `unanimous`, `margin`).

In [ ]:
master = build_master(items, annotations)
multi = master[master["is_multi"]]
disagreements = master[master["disagree"]]
print("master:", master.shape)
master.head()

## §2 — General statistics

In [ ]:
all_labels = pd.concat([master[u].dropna() for u in ANNOTATORS])
print(f"Annotated instances            : {len(master)}")
print(f"  · single-annotator (1 vote)  : {len(master) - len(multi)}")
print(f"  · multi-annotator (>=2)      : {len(multi)}")
print(f"Disagreements (among multi)    : {len(disagreements)}  "
      f"({100 * len(disagreements) / max(len(multi), 1):.0f}% of multi)")
print(f"Global YES rate (base rate)    : "
      f"{100 * (all_labels == 'YES').mean():.0f}%  (on {len(all_labels)} labels)")
print("Majority label:", dict(master["majority"].value_counts()))
print("Category breakdown:", dict(master["category"].value_counts()))

In [ ]:
plots.plot_volume(stats.annotator_volume(master, ANNOTATORS))
plots.plot_majority(master);

In [ ]:
counts = stats.label_counts(master, ANNOTATORS)
display(counts)
plots.plot_label_counts(counts)
plots.plot_label_totals(counts);

## §3 — Per-annotator statistics

Rates and signed bias (gap vs the group mean, in points); NO is the mirror of YES.

In [ ]:
per_ann = stats.label_rates_and_bias(master, ANNOTATORS)
display(per_ann)
plots.plot_rate_and_bias(per_ann, "NO")
plots.plot_rate_and_bias(per_ann, "YES");

### Inter-annotator agreement — Cohen's κ

Computed pairwise on each pair's shared instances
(Landis & Koch scale: see `stats.KAPPA_SCALE`).

In [ ]:
K = stats.pairwise_kappa(master, ANNOTATORS)
plots.plot_kappa_matrix(K)
K.round(3)

In [ ]:
a, b = ANNOTATORS[0], ANNOTATORS[1]
plots.plot_confusion(stats.confusion(master, a, b), a, b);

In [ ]:
show_cards(disagreements, ANNOTATORS, kb,
           title="Disagreements between annotators")

## §4 — Per-category statistics (PERS / ORG / LOC)

In [ ]:
cat = stats.category_summary(master)
display(cat)
plots.plot_category_rates(cat)
plots.plot_agreement_by_category(master);

In [ ]:
# One clear + one borderline example per category
for c in sorted(master["category"].dropna().unique()):
    g = master[master["category"] == c]
    clean = g[g["unanimous"] & g["is_multi"]]
    border = g[g["disagree"]]
    ex = pd.concat([clean.head(1), border.head(1)])
    if len(ex):
        show_cards(ex, ANNOTATORS, kb, img_w=240,
                   title=f"Category {c}: clear vs borderline")

## §5 — Category × annotator

In [ ]:
long = stats.labels_long(master, ANNOTATORS)
display(pd.crosstab([long["annotator"], long["category"]], long["label"]))
plots.plot_labels_by_category(long, ANNOTATORS);

In [ ]:
# Per-category κ is only indicative while n_common is low
display(stats.kappa_per_category(master, ANNOTATORS[0], ANNOTATORS[1]))

## §6 — Extreme cases

Fragile instances are multi-annotated instances whose majority flips if one
vote changes (margin ≤ 1); unanimous lists split by coverage and label.

In [ ]:
fragile = stats.fragile_instances(master)
show_cards(fragile, ANNOTATORS, kb, img_w=260,
           title="Fragile instances (margin <= 1)")

In [ ]:
for cover, label in [("multi", "YES"), ("multi", "NO"),
                     ("mono", "YES"), ("mono", "NO")]:
    is_multi = cover == "multi"
    sub = master[master["unanimous"] & (master["is_multi"] == is_multi)
                 & (master["majority"] == label)]
    if len(sub):
        show_table(sub, ANNOTATORS, title=f"Unanimous {label} — {cover}-annotator")

## §7 — Annotation time (behavioral logs)

Decision time = first `instance_load` → first `annotation_change`, per
(annotator, instance). Outliers are winsorized (Tukey: Q3 + 1.5 IQR) before
any aggregate — see `stats.cap_outliers` for the alternatives.

In [ ]:
times = stats.cap_outliers(load_decision_times(ANNOT_DIR, items))
print(f"{len(times)} timed annotations | cap = {times.attrs['cap_method']} "
      f"@ {times.attrs['cap_threshold']:.0f}s "
      f"| {int(times['is_outlier'].sum())} value(s) capped")
display(stats.timing_summary(times))

In [ ]:
plots.plot_time_matrix(stats.timing_matrix(times))
plots.plot_time_distribution(times);